## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Klasterisasi K-Means & Dendrogram](images/img_09_kmeans_clustering.png)

```
              +------------------------------------------------+
              |           ALUR K-MEANS CLUSTERING              |
              +------------------------------------------------+
                                       |
              +------------------------+-----------------------+
              |                                                |
              v                                                v
      [1] ELBOW METHOD                              [2] SILHOUETTE SCORE
   Inertia / WCSS vs. K                             Kualitas Kohesi & Separasi
   Cari titik tekukan siku                          Maksimumkan nilai s(i) -> 1.0
              |                                                |
              +------------------------+-----------------------+
                                       |
                                       v
                     [3] PROFILING & PERSONA KLASTER
                     Interpretasi Karakteristik Tiap Segmen
```


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_seg = pd.read_csv("../datasets/07_customer_segmentation_clustering.csv")
print("Dataset Segmentasi Pelanggan dimuat. Total baris:", len(df_seg))
display(df_seg.head())


## 🌲 3. Klasterisasi Hirarki & Dendrogram


In [ ]:
feature_cols = ['annual_income_million', 'spending_score_1_100', 'purchase_frequency_yearly', 'tech_savviness_index']
X_raw = df_seg[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Membuat linkage matrix menggunakan Ward's method
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=True, no_labels=True)
plt.title('Dendrogram Klasterisasi Hirarki (Ward Linkage)', fontweight='bold')
plt.xlabel('Indeks Sampel Pengguna')
plt.ylabel('Jarak Euclidean (Ward Distance)')
plt.axhline(y=15, color='r', linestyle='--', label='Garis Potong Klaster (K = 4)')
plt.legend()
plt.show()


## 📐 4. Penentuan K Optimal (Elbow Method & Silhouette Score)


In [ ]:
k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
axes[0].plot(k_range, inertias, 'bo-', lw=2, markersize=8)
axes[0].set_title('Elbow Method (Within-Cluster Sum of Squares)', fontweight='bold')
axes[0].set_xlabel('Jumlah Klaster (K)')
axes[0].set_ylabel('Inertia / WCSS')

# Silhouette Score
axes[1].plot(k_range, sil_scores, 'ro-', lw=2, markersize=8)
axes[1].set_title('Silhouette Score per Nilai K', fontweight='bold')
axes[1].set_xlabel('Jumlah Klaster (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()


## 🏷️ 5. Partisi K-Means (K=4) & Profiling Persona Segmen


In [ ]:
# Melatih model K-Means dengan K=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df_seg['Cluster'] = kmeans_final.fit_predict(X_scaled)

# Profiling rata-rata fitur per klaster
profile = df_seg.groupby('Cluster')[feature_cols].mean().round(2)
profile['Ukuran Segmen (N)'] = df_seg['Cluster'].value_counts().sort_index()

# Labeling Persona
persona_names = {
    0: 'Budget Conscious',
    1: 'Premium VIP High-Spender',
    2: 'Digital Trendsetter',
    3: 'Balanced Mainstream'
}
profile['Nama Persona'] = profile.index.map(persona_names)

print("=== Ringkasan Profiling Karakteristik Klaster ===")
display(profile)

# Visualisasi Scatter Klaster
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_seg, x='annual_income_million', y='spending_score_1_100', hue='Cluster', palette='tab10', s=80, alpha=0.9)
plt.title('Segmentasi Pelanggan Berdasarkan Pendapatan vs. Skor Belanja', fontweight='bold')
plt.xlabel('Pendapatan Tahunan (Juta IDR)')
plt.ylabel('Skor Belanja (1 - 100)')
plt.legend(title='Klaster')
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Mengapa standarisasi fitur wajib dilakukan sebelum K-Means?** K-Means menghitung jarak Euclidean murni. Jika fitur `annual_income_million` bernilai puluhan juta sedangkan `tech_savviness` bernilai puluhan, fitur bernilai besar akan mendominasi jarak dan mendistorsi klaster.

### Data Analysis Key Findings
* Nilai $K=4$ memberikan kombinasi siku *Elbow* dan *Silhouette Score* tertinggi.
* Terbentuk 4 segmen pengguna yang sangat terpisah:
  1. **Budget Conscious**: Pendapatan rendah, skor belanja rendah.
  2. **Premium VIP**: Pendapatan tinggi (>100 juta), belanja sangat tinggi.
  3. **Digital Trendsetter**: Skor kecakapan digital dan belanja tinggi.
  4. **Balanced Mainstream**: Segmen moderat dengan belanja seimbang.

### Insights or Next Steps
* Tim pemasaran dapat merancang promosi terpersonalisasi untuk masing-masing persona klaster.
